In [ ]:
# Custom
import sys
sys.path.insert(0, '../src')
from args import args
import utils
from snippet_creation import SnippetCreation


## Utils and Inference
import csv
from collections import Counter
import json
import numpy as np
import pandas as pd
import re

import os
from tqdm import tqdm

In [ ]:
class args(args):

    setting = ["batching", "single_variable"][1]

    experimental_setting = ["vanilla", "full_source_code", "source_code_grep_chunk", "paragraph_code"][-1]
    
    # static_analyzer_data_path = "../data/static_analyzer_Downloaded_Data/"
    setting = ["batching", "single_variable"][1]
    saving_path = f"../saved_data/inference/{setting}/{experimental_setting}"
    
    variable_column = "variable_name"
    program_path_column = "program_path"
    
    data_path  = ['../data/get_variable_para_info_formatted_info_annotation_set_23_06_2026_with_path.json', '../data/get_variable_para_info_formatted_info_annotation_set_22_06_2026_with_path.json'][0]
    complete_data_path  = ['../data/static_analyzer_Downloaded_Data/all_application_get_variable_para_info_formatted_23_06_2026.json', '../data/get_variable_para_info_formatted_info_annotation_set_22_06_2026_with_path.json'][0]
    
    program_path_column = "corrected_path"

    platform = ["llm_provider_1", "watsonx", "mse", "litellm", "azure"][3]


In [ ]:
import sys
sys.path.insert(0, '..')
from prompts import icl_paragraph_code_prompts

In [ ]:
prompt = icl_paragraph_code_prompts.paragraph_code_prompt_v3_detailed
print(prompt)

In [ ]:
## Check if the saving directories exist or not; If not, then create them
utils.create_save_directories()

In [ ]:
### Check if the experiments saving directories exist or not; If not, then create them
utils.ensure_exp_dirs_exist(args.saving_path)

## Search Source Code

In [ ]:

def read_file_binary_cobol(full_path):
    
    try:
        with open(full_path, "rb") as f:
            content = f.read()

        return {
            "path": full_path,
            "content": content.decode("latin-1")  # bytes → string
        }

    except Exception as e:
        return {
            "error": f"Found file at {full_path} but could not read it: {e}"
        }


    return {"error": "File not found."}



## Gather and Merge Paragraph Context

In [ ]:
def create_variable_definition_snippet(prog_id, app_name, var_name, var_id, file_name):
    """
    Create COBOL variable definition snippets from complete_data_df for a specific program and variable.
    Filters to include only:
    - The variable itself (matching var_name and var_id)
    - Variables that have the considered variable as Father or Ancestor
    
    Args:
        prog_id: Program ID to filter complete_data_df
        app_name: Application name to filter complete_data_df
        var_name: Variable name to filter for (the variable under consideration)
        var_id: Variable ID (VarID) to filter for
        file_name: File name for marking the snippet
    
    Returns:
        String containing marked COBOL variable definition snippet, or empty string if no data found
    """
    try:
        # Filter complete_data_df: ProgID matches prog_id and app_name matches
        program_df = complete_data_df[
            (complete_data_df['ProgID'] == prog_id) & 
            (complete_data_df['app_name'] == app_name)
        ]
        
        if program_df.empty:
            return ""
        
        # Filter for:
        # 1. The variable itself (VarID matches)
        # 2. Variables where Father == var_id
        # 3. Variables where Ancestor contains var_id (if Ancestor is a string/list)
        
        # Handle var_id as either a single value or a list
        if isinstance(var_id, (list, tuple)):
            var_id_list = var_id
        else:
            var_id_list = [var_id]
        
        # Build filter conditions using .isin() for list compatibility
        varid_match = program_df['VarID'].isin(var_id_list)
        father_match = program_df['Father'].isin(var_id_list)
        
        # For Ancestor, check if any var_id appears in the Ancestor field
        ancestor_match = program_df['Ancestor'].apply(
            lambda x: any(str(vid) in str(x) for vid in var_id_list) if pd.notna(x) else False
        )
        
        filtered_df = program_df[varid_match | father_match | ancestor_match]
        
        if filtered_df.empty:
            return ""
        
        # Map complete_data_df columns to snippet_creation.py expected columns
        snippet_df = filtered_df[['VarID', 'VarName', 'PIC', 'szValues', 'iLevel', 'Father']].copy()
        snippet_df.columns = ['var_id', 'var_name', 'pic', 'sz_values', 'i_level', 'father']
        
        # Generate COBOL variable definition snippets
        snippet_creator = SnippetCreation()
        snippets = snippet_creator.create_snippets(snippet_df.drop_duplicates())
        
        if snippets:
            snippet_content = "\n\n".join(snippets.values())
            marked_snippet = f"[start of snippet for {file_name}]\n{snippet_content}\n[end of snippet for {file_name}]"
            return marked_snippet
        
        return ""
    
    except Exception as e:
        print(f"Warning: Could not create snippet for prog_id={prog_id}, app_name={app_name}, var_name={var_name}, var_id={var_id}: {e}")
        return ""


def gather_context(corrected_paths, curr_StartRows, curr_EndRows, curr_ParaNames, var_name, app_name, curr_prog_ids=None, var_id=None):
    """
    Read files and extract paragraphs with markers.
    If var_name not found in paragraphs, generate COBOL variable definition snippets from complete_data_df.
    
    Data Schema:
    - data_df columns: variable_name, app_name, ProgID, VarID, corrected_path, ParaName, ParaStartRow, ParaEndRow
    - complete_data_df columns: ProgID, app_name, VarName, VarID, Father, Ancestor, PIC, szValues, iLevel
    
    Args:
        corrected_paths: List of file paths from data_df['corrected_path']
        curr_StartRows: List of paragraph start rows from data_df['ParaStartRow']
        curr_EndRows: List of paragraph end rows from data_df['ParaEndRow']
        curr_ParaNames: List of paragraph names from data_df['ParaName']
        var_name: Variable name from data_df['variable_name']
        app_name: Application name from data_df['app_name']
        curr_prog_ids: List of program IDs from data_df['ProgID']
        var_id: Variable ID from data_df['VarID']
    """
    source_codes = []
    
    # Group by file path to avoid re-reading files
    file_to_paragraphs = {}
    file_to_prog_id = {}
    
    for idx, (file_path, start, end, para_name) in enumerate(zip(corrected_paths, curr_StartRows, curr_EndRows, curr_ParaNames)):
        if file_path not in file_to_paragraphs:
            file_to_paragraphs[file_path] = []
            if curr_prog_ids is not None and idx < len(curr_prog_ids):
                file_to_prog_id[file_path] = curr_prog_ids[idx]
        
        entry = (int(start), int(end), para_name)
        if entry not in file_to_paragraphs[file_path]:
            file_to_paragraphs[file_path].append(entry)
    
    # Process each file
    for file_path, paragraphs in file_to_paragraphs.items():
        file_content = read_file_binary_cobol(full_path=file_path)
        
        if 'error' in file_content:
            continue
        
        file_name = os.path.basename(file_path)
        prog_id = file_to_prog_id.get(file_path)
        
        source_code = file_content['content']
        lines = source_code.splitlines()
        
        # Extract paragraphs with markers
        paragraph_sources = []
        for start, end, para_name in paragraphs:
            if start <= len(lines) and end <= len(lines):
                snippet = lines[start-1:end]
                marked_para = f"[start of {para_name}]\n" + "\n".join(snippet) + f"\n[end of {para_name}]"
                paragraph_sources.append(marked_para)
        
        file_content_str = "\n".join(paragraph_sources)
        
        # If variable not in paragraphs, add COBOL definition snippets from complete_data_df
        if var_name and var_name not in file_content_str and prog_id is not None:
            snippet_str = create_variable_definition_snippet(prog_id, app_name, var_name, var_id, file_name)
            # snippet_str = " "
            if snippet_str:
                file_content_str = snippet_str + "\n" + file_content_str
        
        marked_content = f"[start of {file_name}]\n{file_content_str}\n[end of {file_name}]"
        source_codes.append(marked_content)
    
    return source_codes


def merge_context(source_codes):
    return "\n\n".join(source_codes)


### Data Preprocessing

In [ ]:
data_df = pd.read_json(args.data_path)

In [ ]:
variable_names = list(data_df[args.variable_column])

In [ ]:
complete_data_df = pd.read_json(args.complete_data_path)

### Validating and Loading the Inference Pipeline

In [ ]:
inference = utils.Inference(args)

In [ ]:
model_names = utils.validate_api_endpoints(args, args.platform)

print(f"Models from {args.platform}: {model_names}\n" )


model_names = ['llama3_3_70b', 'gpt_oss_120b', 'gpt_oss_20b', 'qwen_3_8B', 'llama4_17b_16e', 'qwen_3_30b_a3b_thinking', 'gemma_4_31B_it']

print(f"Selected Models from {args.platform}: {model_names}" )

In [ ]:
pred_folder=args.saving_path+"/"
pred_folder

In [ ]:

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

def write_json_record(file_path: str, record: dict) -> None:
    """Append a record to a JSON array file."""
    with open(file_path, 'r+') as jsonfile:
        data = json.load(jsonfile)
        data.append(record)
        jsonfile.seek(0)
        json.dump(data, jsonfile, indent=2, cls=NumpyEncoder)
        jsonfile.truncate()

In [ ]:
file_names = []
for model_name in tqdm(model_names, desc="Models", position=0):
    prediction_df = data_df
    
    file_name = f"{model_name}_variable_DD_{args.experimental_setting}_.json"
    file_name = utils.update_file_name(
        file_name, pred_folder=pred_folder, extension=".json"
    )
    # Checkpointing Logic- model_name can be a string representing model identifier. 
    # The model_name and file_name are intended to be manually defined to resume checkpointing. This assume that the file is already created.
    
    if model_name == '':
        file_name = 'mistral_medium_variable_DD_icl_source_code_1.json'
        checkpoint_index = 0

    else:
        checkpoint_index = 0
        fields = list(prediction_df.columns) + ["prompt_used", "token_statistics", "predictions_DD"]

        with open(pred_folder + file_name, 'w') as jsonfile:
            json.dump([], jsonfile)

    print(f"Starting Inference for {model_name} in {file_name}")
    full_path = pred_folder + file_name

    pbar = tqdm(
        total=len(variable_names),
        desc=f"Inference [{model_name}]",
        position=1,
        leave=False,
        dynamic_ncols=True
    )
    
    for i, var_name in enumerate(variable_names):
        if i < checkpoint_index:
            pbar.update(1)
            continue
        if i == checkpoint_index:
            print(f"starting inference from index {i}")
        row_data = list(prediction_df.iloc[i])
        
        
        curr_ParaStartRows = prediction_df.iloc[i]['ParaStartRow']
        curr_ParaEndRows = prediction_df.iloc[i]['ParaEndRow']
        curr_ParaNames = prediction_df.iloc[i]['ParaName']
        
        corrected_paths = prediction_df.iloc[i][args.program_path_column]
        app_name = prediction_df.iloc[i]['app_name']
        curr_prog_ids = prediction_df.iloc[i]['ProgID']
        var_id = prediction_df.iloc[i]['VarID']
        
        # Fetch Prompt
        current_prompt = prompt[:]
        # Add Variable in the template
        current_prompt = current_prompt.replace("{{TEST_VARIABLE}}", f"{var_name}")
        # Add Source Code in the template
        try:
            source_codes = gather_context(
                corrected_paths=corrected_paths,
                curr_StartRows=curr_ParaStartRows,
                curr_EndRows=curr_ParaEndRows,
                curr_ParaNames=curr_ParaNames,
                var_name=var_name,
                app_name=app_name,
                curr_prog_ids=curr_prog_ids,
                var_id=var_id
            )
            source_code_string = merge_context(source_codes)

        except Exception as e:
            # Don't perform inference if source code is not found
            response = f"error found: {e}"
            token_stats = f"error found: {e}"
            record = dict(zip(fields, row_data + ["", token_stats, response]))
            
            write_json_record(full_path, record)
            pbar.update(1)
            continue
        
        try:
            # Attempt Inference if source code is available
            current_prompt = current_prompt.replace("{{SOURCE_CONTEXT}}", f"{source_code_string}")
            
            response, raw_output = inference(
                prompt=current_prompt,
                inference_platform=args.platform,
                model_name=model_name,
                return_raw=True
            )
            
            token_stats = dict(dict(raw_output)['usage'])

            # Filtering For serialization
            token_stats = {k: token_stats[k] for k in ('completion_tokens', 'prompt_tokens', 'total_tokens')}
            
        except Exception as e:
            if not response and not token_stats:
                response = token_stats = f"error found: {e}"

            else: 
                response = str(response)
                token_stats = str(response)

        record = dict(zip(fields, row_data + [current_prompt, token_stats, response]))
        write_json_record(full_path, record)
        pbar.update(1)

    #     break

    # break

    pbar.close()
    file_names.append(file_name)